# Week 5 Lab — Raster & Vector–Raster Integration in Vienna

**Guided A:** four raster operations using a small, **new** synthetic example.  
**Guided B:** calculate **Built-up share for one real Vienna district** from ESA WorldCover 2021 (with optional map views).  
**Independent checkpoint:** calculate **Tree Cover share for all 23 Vienna districts**, then inspect the ready-made map.

**How to work:** follow **load/create → inspect properties → optional quick plot → analyze → check results**. All plotting code is provided; every plot is **Optional** and may be skipped.

## Before you start — Moodle files and data sources

You should receive this notebook and two data files **together in the same folder**:

- `W05_Vienna_WorldCover2021_10m.tif` — **full Vienna extent**, 2021 ESA WorldCover 10 m category raster, cropped and nearest-neighbour reprojected to EPSG:31256.
- `W05_Vienna_Districts_23.geojson` — all **23** Vienna district polygons, supplied in EPSG:4326 and reprojected in this notebook.

Keep the GeoTIFF and GeoJSON in the **same folder as this notebook**. If they are missing, ask the instructor for the **complete** Week 5 data package. **Do not substitute Week 4's small `vienna_worldcover_clip.tif` or Week 3's four-district file**: those cannot answer the full-city question.

**Sources:** ESA WorldCover 2021 v200 (10 m), © ESA WorldCover project 2021 / Contains modified Copernicus Sentinel data (2021) processed by ESA WorldCover consortium; Stadt Wien Open Government Data, district boundaries, CC BY 4.0. Class code **10 = Tree Cover**, **50 = Built-up**, **0 = NoData**. Raster cell classifications are not ground-truth measurements.

These files have already been prepared for class. You do not need to download or process the original source tiles.

## 0. Import packages and set up the workspace

In [ ]:
# Path helps us refer to local files using simple filenames.
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
# Rasterio reads/writes rasters; GeoPandas handles vector geometry and tables.
import rasterio
from rasterio.transform import from_origin
from rasterio.mask import mask
from shapely.geometry import Point, box, mapping

print('Packages imported. Ready to work with raster and vector data.')

**Check the workspace.** Keep this notebook and the two supplied Vienna input files in the same folder. The synthetic exercises create a few tiny `W05_Guided_*.tif` teaching files beside the notebook.

In [ ]:
workdir = Path('.')
data_dir = Path('.')
print('Current folder:', Path.cwd())
print('Data and notebook use the same folder.')
print('Expected input files:')
print('- W05_Vienna_WorldCover2021_10m.tif')
print('- W05_Vienna_Districts_23.geojson')

## Guided A — Four operations with synthetic raster data

**Predict → run → inspect → explain.** These values deliberately differ from the Lecture example. Work through each short cell and look at its output before going on. Every plotting cell is **Optional** and already written for you.

### A1. Raster alignment — do neighboring tiles really touch?

Two north-up grids have the same CRS (`EPSG:32633`) and pixel size (50 m), but different horizontal dimensions. A is **2 × 3** and B is **2 × 2**. Before running the calculation, predict A's right edge and whether a shifted B produces a gap.

**Step 1 — Create the two small tiles.** Inspect both values and shapes before considering their locations.

In [ ]:
# Each row is a raster row; each number is one cell value.
# NumPy arrays hold values only; their geographic positions come later.
tile_a = np.array([[2, 4, 6], [8, 10, 12]], dtype='float32')
tile_b = np.array([[14, 16], [18, 20]], dtype='float32')

print('Tile A:\n', tile_a)
print('Tile A shape:', tile_a.shape)
print('Tile B:\n', tile_b)
print('Tile B shape:', tile_b.shape)

**Step 2 — Give each tile a geographic position.** The aligned version of B begins where A ends. The shifted version starts farther east.

In [ ]:
# from_origin(left, top, pixel_width, pixel_height)
transform_a = from_origin(400000, 5350000, 50, 50)
# A has 3 columns × 50 m = 150 m, so aligned B starts at x=400150.
transform_b_aligned = from_origin(400150, 5350000, 50, 50)
transform_b_shifted = from_origin(400175, 5350000, 50, 50)
print('Tile A position:', transform_a)
print('Tile B (aligned) position:', transform_b_aligned)

**Save the teaching tiles.** This short setup puts each array and its transform into a small GeoTIFF. We can then inspect the actual files in the same way as in the lecture.

In [ ]:
path_a = workdir / 'W05_Guided_Tile_A.tif'
path_b = workdir / 'W05_Guided_Tile_B.tif'
path_b_shifted = workdir / 'W05_Guided_Tile_B_Shifted.tif'

for path, values, transform in [
    (path_a, tile_a, transform_a),
    (path_b, tile_b, transform_b_aligned),
    (path_b_shifted, tile_b, transform_b_shifted)
]:
    with rasterio.open(path, 'w', driver='GTiff',
                       height=values.shape[0], width=values.shape[1],
                       count=1, dtype=values.dtype,
                       crs='EPSG:32633', transform=transform) as dst:
        dst.write(values, 1)
print('Created the three teaching tile files.')


**Check the grids first.** Both CRSs must be defined and match, the cell sizes must match, and this simple join uses unrotated, north-up grids. `np.allclose()` is a NumPy function. Here `rtol=0` turns off the relative tolerance and `atol=1e-6` allows only a tiny absolute difference in the coordinate units. This matters because a default relative tolerance can hide a real offset in large map coordinates.

In [ ]:
with rasterio.open(path_a) as src_a, rasterio.open(path_b) as src_b:
    same_crs = src_a.crs is not None and src_a.crs == src_b.crs
    same_resolution = np.allclose(src_a.res, src_b.res, rtol=0, atol=1e-6)
    north_up = all(
        src.transform.b == 0 and src.transform.d == 0
        and src.transform.a > 0 and src.transform.e < 0
        for src in (src_a, src_b)
    )

print('Same defined CRS:', same_crs)
print('Same cell size:', same_resolution)
print('Both grids unrotated and north-up:', north_up)
assert same_crs and same_resolution and north_up


**Read the boundaries from the files.** `src.bounds` gives the left, bottom, right and top coordinates. Compare A's right edge with the two versions of B's left edge.

In [ ]:
with rasterio.open(path_a) as src_a, rasterio.open(path_b) as src_b:
    a = src_a.bounds
    b = src_b.bounds
with rasterio.open(path_b_shifted) as src_shifted:
    b_shifted = src_shifted.bounds

right_a = a.right
left_b = b.left
left_b_shifted = b_shifted.left
print('A right edge:', right_a)
print('B aligned left edge:', left_b)
print('B shifted left edge:', left_b_shifted)
print('Gap if B is shifted (m):', left_b_shifted - right_a)
print('Do these two edge coordinates match?',
      np.allclose(a.right, b.left, rtol=0, atol=1e-6))


### A1 visual check — Optional: aligned tiles versus a gap

The two panels use the **same Tile A and Tile B values**, but B starts at a different x-coordinate. Notice the **25 m gap** in the shifted case: `np.hstack()` would not detect this. **Run or skip; no plotting code to write.**

In [ ]:
# OPTIONAL PLOT — visualize the positions checked above; no new calculation is required.
# Easting is relative to A's left edge; northing is relative to A's bottom edge.
a_width = tile_a.shape[1] * transform_a.a
b_width = tile_b.shape[1] * transform_b_aligned.a
tile_height = tile_a.shape[0] * abs(transform_a.e)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
for ax, b_transform, title in zip(
    axes,
    [transform_b_aligned, transform_b_shifted],
    ['Aligned: no gap', 'Shifted: 25 m gap']
):
    # The first tile always starts at 0; B's start depends on its transform.
    b_start = b_transform.c - transform_a.c
    ax.imshow(tile_a, extent=(0, a_width, 0, tile_height),
              origin='upper', cmap='viridis', vmin=2, vmax=20,
              interpolation='nearest')
    ax.imshow(tile_b, extent=(b_start, b_start + b_width, 0, tile_height),
              origin='upper', cmap='viridis', vmin=2, vmax=20,
              interpolation='nearest')

    # The dashed line shows where Tile A ends.
    ax.axvline(a_width, color='#c0392b', linestyle='--', linewidth=2)
    if b_start > a_width:
        # Light red shows the area with NO raster cells between the tiles.
        ax.axvspan(a_width, b_start, color='#ef9a9a', alpha=0.8)
    ax.text(a_width / 2, tile_height / 2, 'A', ha='center', va='center',
            color='white', fontsize=14, weight='bold')
    ax.text(b_start + b_width / 2, tile_height / 2, 'B', ha='center',
            va='center', color='#1c2230', fontsize=14, weight='bold')
    ax.set(title=title, xlabel='Distance east of A origin (m)',
           ylabel='Distance north of A bottom (m)')
    ax.set_xlim(-5, a_width + b_width + 35)
    ax.set_ylim(-5, tile_height + 5)
    ax.set_aspect('equal')  # Equal distances have equal lengths on the plot.
plt.show()

**Self-check:** Does matching `shape` prove geographic adjacency? Why would `np.hstack()` still run for the shifted case?

**Left or right, step 1 — Match the top and bottom.** A touching edge only gives a direct horizontal join when both tiles cover the same full vertical range.

In [ ]:
same_y = np.allclose(
    [a.bottom, a.top], [b.bottom, b.top], rtol=0, atol=1e-6)
print('Same top and bottom:', same_y)


**Left or right, step 2 — Check which edges meet.** Each direction describes B relative to A.

In [ ]:
east = same_y and np.allclose(a.right, b.left, rtol=0, atol=1e-6)
west = same_y and np.allclose(a.left, b.right, rtol=0, atol=1e-6)
print('B is directly east of A:', east)
print('B is directly west of A:', west)


**Above or below — Use the same idea in the other direction.** First match the left and right boundaries, then compare the touching top and bottom edges.

In [ ]:
same_x = np.allclose(
    [a.left, a.right], [b.left, b.right], rtol=0, atol=1e-6)
north = same_x and np.allclose(a.top, b.bottom, rtol=0, atol=1e-6)
south = same_x and np.allclose(a.bottom, b.top, rtol=0, atol=1e-6)
print('B is directly north of A:', north)
print('B is directly south of A:', south)


**Check the gap case.** The shifted tile still has the same top and bottom, but its left edge no longer meets A's right edge. An all-False direction result means that this simple full-edge join does not apply.

In [ ]:
shifted_same_y = np.allclose(
    [a.bottom, a.top], [b_shifted.bottom, b_shifted.top], rtol=0, atol=1e-6)
shifted_east = shifted_same_y and np.allclose(
    a.right, b_shifted.left, rtol=0, atol=1e-6)
print('Shifted B is a direct east neighbour:', shifted_east)
assert (east, west, north, south) == (True, False, False, False)
assert not shifted_east


### A2. Mosaic — join two checked tiles

The correct B is directly east of A. Predict the output array shape and the values of the first and last columns. NumPy does **not** check CRS, transforms, or gaps.

**Step 1 — Join the arrays.** We already checked the geographic placement; NumPy itself does not know about it.

In [ ]:
assert east, 'Check that B is directly east of A before joining these arrays.'
# Put the columns of B to the right of the columns of A.
mosaic = np.hstack([tile_a, tile_b])
print('Mosaic values:\n', mosaic)

**Step 2 — Inspect the result.** Did the number of rows stay the same? Did the number of columns increase?

In [ ]:
print('Mosaic shape:', mosaic.shape)
print('First column:', mosaic[:, 0])
print('Last column:', mosaic[:, -1])
print('Expected shape is (2, 5). Does it match?', mosaic.shape == (2, 5))

### A2 visual check — Optional: two inputs become one array

Compare the **rows and columns** before and after `np.hstack()`. Both inputs keep their cell values; only the horizontal array shape changes. The separate geographic mosaic map below shows where the result would be placed. **Run or skip; no plotting code to write.**

In [ ]:
# OPTIONAL PLOT — show the input arrays and the combined output side by side.
# Each square corresponds to one cell, and the printed numbers are its values.
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.1), constrained_layout=True)
for ax, (label, grid) in zip(
    axes,
    [('Tile A (2 × 3)', tile_a), ('Tile B (2 × 2)', tile_b),
     ('Mosaic (2 × 5)', mosaic)]
):
    ax.imshow(grid, origin='upper', cmap='viridis', vmin=2, vmax=20,
              interpolation='nearest')
    # The text makes it easy to check that no values moved between rows.
    for row in range(grid.shape[0]):
        for col in range(grid.shape[1]):
            ax.text(col, row, f'{grid[row, col]:g}', ha='center', va='center',
                    color='#1c2230' if grid[row, col] >= 13 else 'white',
                    fontsize=10, weight='bold')
    ax.set_title(label)
    ax.set_xticks(range(grid.shape[1]))
    ax.set_yticks(range(grid.shape[0]))
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    # White grid lines outline the individual array cells.
    ax.set_xticks(np.arange(-0.5, grid.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, grid.shape[0], 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=1)
    ax.tick_params(which='minor', length=0)
plt.show()

### A2 visual check — Optional: see the raster mosaic

The colours show synthetic cell values. The dashed line marks where Tile A ends and Tile B begins. This is a **georeferenced toy grid**, not a real satellite image. **Run or skip; no plotting code to write.**

In [ ]:
# OPTIONAL PLOT — this cell only displays the mosaic; you can skip it.
# Matplotlib uses 'fig' for the whole figure and 'ax' for its plotting area.
# Calculate the outer map coordinates from the grid origin and pixel size.
left = transform_a.c
top = transform_a.f
right = left + mosaic.shape[1] * transform_a.a
bottom = top + mosaic.shape[0] * transform_a.e

fig, ax = plt.subplots(figsize=(8.2, 3.8))
# imshow() turns the array into a coloured grid.
# extent gives its real map boundaries; origin='upper' places row 0 at the top.
image = ax.imshow(mosaic, extent=(left, right, bottom, top), origin='upper',
                  cmap='viridis', interpolation='nearest', aspect='equal')

# Mark the exact x-coordinate where Tile A meets Tile B.
ax.axvline(right_a, color='white', linestyle='--', linewidth=2, label='Tile boundary')
ax.text(left + 70, top - 12, 'Tile A', color='white', weight='bold', ha='center')
ax.text(right_a + 50, top - 12, 'Tile B', color='white', weight='bold', ha='center')

# Axis labels describe map coordinates; the legend explains the dashed line.
ax.set(xlabel='Easting (m, toy CRS)', ylabel='Northing (m, toy CRS)',
       title='Optional — synthetic mosaic: 2 tiles → 1 wider raster')
ax.ticklabel_format(style='plain', useOffset=False)
ax.legend(loc='lower right')
# The colour bar explains how colours correspond to the synthetic cell values.
fig.colorbar(image, ax=ax, label='Synthetic values', shrink=0.85)
plt.tight_layout()  # Keep the labels within the figure.
plt.show()          # Display the optional figure.

**Optional connection — A mosaic can preserve a gap.** A georeferenced mosaic can keep uncovered areas as NoData. If the input cells do not line up on a common grid, as in our half-cell-shifted B, an output grid and any required resampling must be chosen first. `np.hstack()` only joins array columns; it does not handle that geographic placement.

### A3. Difference — retain missing cells

Now use **a different, independent** 4 × 4 pair of surfaces. One input is missing at row 1 / column 1; the other at row 2 / column 2. Predict the valid-cell count and the first difference before running.

**Step 1 — Create the two surfaces.** These are independent of the preceding Mosaic example. Compare where each input has NoData.

In [ ]:
nodata = -9999.0
surface_1 = np.array([
    [11, 13, 15, 17],
    [19, nodata, 23, 25],
    [27, 29, 31, 33],
    [35, 37, 39, 41]
], dtype='float32')
surface_2 = np.array([
    [9, 12, 14, 16],
    [18, 20, 22, 24],
    [26, 28, nodata, 32],
    [34, 36, 38, 39]
], dtype='float32')

print('Surface 1:\n', surface_1)
print('Surface 2:\n', surface_2)
print('Both surfaces have the same shape:', surface_1.shape == surface_2.shape)

**Before subtracting, check the spatial grid.** We save the two synthetic surfaces with geographic metadata, then check their CRS, resolution, shape and transform. Having the same array shape alone does not show that corresponding cells occupy the same place.

In [ ]:
toy_transform = from_origin(500000, 5340000, 50, 50)
toy_raster_path = workdir / 'W05_Guided_Surface.tif'
surface_2_path = workdir / 'W05_Guided_Surface_2.tif'

for path, values in [(toy_raster_path, surface_1), (surface_2_path, surface_2)]:
    with rasterio.open(path, 'w', driver='GTiff',
                       width=values.shape[1], height=values.shape[0],
                       count=1, dtype=values.dtype, crs='EPSG:32633',
                       transform=toy_transform, nodata=nodata) as dst:
        dst.write(values, 1)


In [ ]:
with rasterio.open(toy_raster_path) as src_1, rasterio.open(surface_2_path) as src_2:
    same_crs = src_1.crs is not None and src_1.crs == src_2.crs
    same_resolution = np.allclose(src_1.res, src_2.res, rtol=0, atol=1e-6)
    same_shape = src_1.shape == src_2.shape
    same_transform = np.allclose(src_1.transform, src_2.transform, rtol=0, atol=1e-6)

print('Same defined CRS:', same_crs)
print('Same cell size:', same_resolution)
print('Same rows and columns:', same_shape)
print('Same grid placement and orientation:', same_transform)
assert same_crs and same_resolution and same_shape and same_transform


**Also check what the values mean.** In this example, both surfaces represent the same synthetic quantity in the same units at two times. We know this from how the data were defined. Python cannot infer scientific comparability from the shape or CRS. Subtracting temperature from elevation would still be meaningless even if all four grid checks passed.

**Step 2 — Find valid cells in both surfaces.** `True` means that both values can be used for the difference.

In [ ]:
# A cell can be compared only when neither raster contains NoData.
# & means BOTH surfaces must have a valid value at the same cell.
valid = (surface_1 != nodata) & (surface_2 != nodata)
print('Valid-cell mask (True = usable):\n', valid)
print('Number of valid cells:', valid.sum())

### A3 visual check — Optional: why do we need a valid-cell mask?

The first two grids identify usable pixels in each input. The last grid shows the cells that are **valid in both rasters**. A cell is included in the Difference only when **both** inputs are valid. **Run or skip; no plotting code to write.**

In [ ]:
# OPTIONAL PLOT — green means usable; light red means a NoData cell.
# This view explains the Boolean mask BEFORE subtracting raster values.
from matplotlib.colors import ListedColormap

usable_1 = surface_1 != nodata
usable_2 = surface_2 != nodata
fig, axes = plt.subplots(1, 3, figsize=(10, 3.3), constrained_layout=True)
for ax, (label, usable) in zip(
    axes,
    [('Surface 1 valid cells', usable_1),
     ('Surface 2 valid cells', usable_2),
     ('Both valid (used for Difference)', valid)]
):
    # Boolean False becomes the first colour; True becomes the second colour.
    ax.imshow(usable.astype(int), cmap=ListedColormap(['#edb2a5', '#79b99a']),
              vmin=0, vmax=1, origin='upper', interpolation='nearest')
    for row in range(usable.shape[0]):
        for col in range(usable.shape[1]):
            ax.text(col, row, 'OK' if usable[row, col] else 'X',
                    ha='center', va='center', fontsize=9)
    ax.set(title=label, xlabel='Column', ylabel='Row')
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticks(np.arange(-0.5, 4, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 4, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=1)
    ax.tick_params(which='minor', length=0)
plt.show()
print('Cells shared by both rasters:', int(valid.sum()), 'of', valid.size)

**Step 3 — Calculate the difference.** Begin with NoData everywhere, then fill only the valid positions.

In [ ]:
# Start with a missing result everywhere. Only valid cells are filled below.
difference = np.full(surface_1.shape, nodata, dtype='float32')
difference[valid] = surface_1[valid] - surface_2[valid]

print('Difference raster:\n', difference)
print('First cell: 11 - 9 =', difference[0, 0])
print('Missing cells are still NoData:', difference[1, 1], difference[2, 2])

### A3 visual check — Optional: compare the two surfaces and their difference

View the three **synthetic grids** side by side. Grey squares are NoData. These grids do **not** represent real Vienna observations, and their axes are raster rows/columns rather than map coordinates. **Run or skip; no plotting code to write.**

In [ ]:
# OPTIONAL PLOT — shows three toy arrays, NOT a map of real locations.
# subplots(1, 3) creates one row with three plotting areas ('axes').
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5), constrained_layout=True)
inputs = [(surface_1, 'Surface 1'), (surface_2, 'Surface 2'),
          (difference, 'Surface 1 − Surface 2')]

# The loop draws one array in each subplot, from left to right.
for ax, (array, title) in zip(axes, inputs):
    # Hide the -9999 NoData cells instead of colouring them as normal values.
    cell_values = np.ma.masked_equal(array, nodata)
    colors = plt.get_cmap('viridis' if title != 'Surface 1 − Surface 2' else 'RdBu_r').copy()
    colors.set_bad('#d7d7d7')  # Grey means missing, not a measured value.

    # Use the same colour range for both input surfaces to compare them fairly.
    # The difference gets its own colour scale centred on zero.
    if title == 'Surface 1 − Surface 2':
        ax.imshow(cell_values, cmap=colors, vmin=-3, vmax=3, interpolation='nearest')
    else:
        ax.imshow(cell_values, cmap=colors, vmin=9, vmax=41, interpolation='nearest')

    # These axes show row/column indices, not geographic coordinates.
    ax.set(title=title, xlabel='Column', ylabel='Row')
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    # Minor ticks lie between cells; their white grid lines outline each cell.
    ax.set_xticks(np.arange(-.5, 4, 1), minor=True)
    ax.set_yticks(np.arange(-.5, 4, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=1)
    ax.tick_params(which='minor', length=0)
plt.show()

**Why a mask?** Both inputs must be valid at the *same* location. `-9999` is a NoData marker here, whereas 0 could be a legitimate observation in another dataset.

### A4. Inspect the raster before using points and polygons

**Reopen the saved first surface.** We created this GeoTIFF before the difference calculation. Its values now have a CRS, cell size, map bounds and NoData value, which we need for the point and polygon operations.

In [ ]:
with rasterio.open(toy_raster_path) as src:
    print('CRS:', src.crs)
    print('Resolution:', src.res)
    print('Bounds:', src.bounds)
    print('NoData value:', src.nodata)

### A4 quick look — Optional: plot the GeoTIFF immediately after loading it

A useful routine is **open the data → inspect its properties → plot it** before doing any analysis. This simple map uses the GeoTIFF's *real map bounds* instead of row and column numbers. Can you find the NoData cell? **Optional: run or skip; all plotting code is provided.**

In [ ]:
# OPTIONAL QUICK LOOK — visually check the saved GeoTIFF before sampling it.
# The raster now contains a CRS and transform, not just array values.
with rasterio.open(toy_raster_path) as src:
    raster_image = src.read(1, masked=True)  # Mask -9999 instead of showing it as a value.
    raster_bounds = src.bounds

# Draw the pixels at their map coordinates (left, right, bottom, top).
quick_cmap = plt.get_cmap('viridis').copy()
quick_cmap.set_bad('#dddddd')  # Grey squares represent missing data.
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(raster_image,
                  extent=(raster_bounds.left, raster_bounds.right,
                          raster_bounds.bottom, raster_bounds.top),
                  origin='upper', cmap=quick_cmap, interpolation='nearest')

# Label the tiny grid so you can compare the map with the original array.
for row in range(raster_image.shape[0]):
    for col in range(raster_image.shape[1]):
        x = raster_bounds.left + (col + 0.5) * 50
        y = raster_bounds.top - (row + 0.5) * 50
        missing = bool(np.ma.getmaskarray(raster_image)[row, col])
        label = 'NoData' if missing else f'{raster_image[row, col]:g}'
        ax.text(x, y, label, ha='center', va='center', fontsize=8,
                color='black' if missing or raster_image[row, col] >= 27 else 'white')

ax.set(title='Optional — first look at the saved raster',
       xlabel='Easting (m)', ylabel='Northing (m)')
ax.xaxis.set_major_locator(MaxNLocator(3))
ax.yaxis.set_major_locator(MaxNLocator(4))
ax.ticklabel_format(style='plain', useOffset=False)
fig.colorbar(image, ax=ax, label='Synthetic cell value', shrink=0.8)
plt.tight_layout()
plt.show()


### A5. Point sampling — from coordinates to raster values

First locate P2 on the grid by hand. The upper-left origin is (500000, 5340000), and each pixel is 50 m wide. Then use Rasterio to sample all three points. Like Week 3, `Point` geometry and the CRS live in a GeoDataFrame.

**Step 1 — Create the Point features.** The `geometry` column contains map coordinates, not row/column numbers.

In [ ]:
# A GeoDataFrame stores both point IDs and map locations in one table.
points = gpd.GeoDataFrame(
    {'point_id': ['P1', 'P2', 'P3']},
    geometry=[Point(500025, 5339975),
              Point(500125, 5339875),
              Point(500175, 5339825)],
    crs='EPSG:32633'
)
print('Points CRS:', points.crs)
display(points)

### A5 quick look — Optional: where are the points before sampling?

Before adding raster values to the point table, **plot the point geometries**. Check whether all three points fall within the raster's extent. This plot shows the point **locations only**, not sampled results. **Optional: no code to write.**

In [ ]:
# OPTIONAL QUICK LOOK — check the input Point geometry before extracting cell values.
from matplotlib.patches import Rectangle

with rasterio.open(toy_raster_path) as src:
    raster_bounds = src.bounds

fig, ax = plt.subplots(figsize=(6, 5))
# The rectangle is the raster footprint; it is not the raster's pixel values.
ax.add_patch(Rectangle((raster_bounds.left, raster_bounds.bottom),
                       raster_bounds.right - raster_bounds.left,
                       raster_bounds.top - raster_bounds.bottom,
                       facecolor='#e8edf1', edgecolor='#566775', linewidth=1.3))
# GeoPandas uses each Point's geometry coordinates automatically.
points.plot(ax=ax, color='#c34f45', markersize=60)
for _, point in points.iterrows():
    ax.annotate(point['point_id'],
                (point.geometry.x, point.geometry.y),
                xytext=(5, 5), textcoords='offset points', fontsize=10)

ax.set_xlim(raster_bounds.left - 15, raster_bounds.right + 15)
ax.set_ylim(raster_bounds.bottom - 15, raster_bounds.top + 15)
ax.set_aspect('equal')
ax.set(title='Optional — point locations before sampling',
       xlabel='Easting (m)', ylabel='Northing (m)')
ax.ticklabel_format(style='plain', useOffset=False)
plt.tight_layout()
plt.show()


**Step 2 — Take out the coordinates and sample the raster.** Predict P2’s value before you run the sampling cell.

In [ ]:
# Coordinates must use the same CRS as the raster.
coords = [(point.x, point.y) for point in points.geometry]
print('Coordinates to sample:', coords)

with rasterio.open(toy_raster_path) as src:
    assert points.crs == src.crs, 'Match the point and raster CRS first.'
    samples = list(src.sample(coords, indexes=1, masked=True))

# Take band 1 from each sample. A masked sample stays missing.
sampled = [np.nan if np.ma.is_masked(value[0]) else float(value[0])
           for value in samples]
print('Values returned by the raster:', sampled)


**Step 3 — Add the sampled values to the Point table.** Now each geographic feature has a new attribute.

In [ ]:
# Copy the vector table so the sample values become new point attributes.
point_results = points.copy()
point_results['raster_value'] = sampled
display(point_results[['point_id', 'raster_value']])
print('Does P2 have the predicted value 31?', point_results.loc[1, 'raster_value'] == 31)

### A5 visual check — Optional: where do the sampled points fall?

The point markers appear on the **georeferenced synthetic raster**. The labels show the values already obtained with `src.sample()`. Notice that a point returns the value of its containing cell. **Run or skip; no plotting code to write.**

In [ ]:
# OPTIONAL PLOT — the sampling results are already calculated; this only shows them.
# Read the georeferenced raster: its bounds connect array pixels to map coordinates.
with rasterio.open(toy_raster_path) as src:
    bounds = src.bounds
    toy_pixels = src.read(1, masked=True)  # Keep NoData separate from valid values.

fig, ax = plt.subplots(figsize=(6.5, 5.5))
# imshow() draws the raster, and extent positions it at the correct map coordinates.
image = ax.imshow(toy_pixels,
                  extent=(bounds.left, bounds.right, bounds.bottom, bounds.top),
                  origin='upper', cmap='viridis', interpolation='nearest')

# scatter() places the vector points on the same map using their x/y coordinates.
# White circles with red borders stand out against the coloured raster.
ax.scatter(point_results.geometry.x, point_results.geometry.y,
           s=100, marker='o', facecolor='white', edgecolor='#ba362c',
           linewidth=1.8, zorder=3)

# Label each Point with its ID and the value returned by src.sample().
# xytext is a small display offset so the label does not cover the circle.
for _, p in point_results.iterrows():
    near_right = p.geometry.x > bounds.left + 0.75 * (bounds.right - bounds.left)
    ax.annotate(f"{p['point_id']}: {p['raster_value']:g}",
                (p.geometry.x, p.geometry.y), xytext=(-7, 5) if near_right else (7, 5),
                ha='right' if near_right else 'left', textcoords='offset points', color='#702015', fontsize=9, weight='bold',
                bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.85, 'pad': 1})

ax.set(xlabel='Easting (m, toy CRS)', ylabel='Northing (m, toy CRS)',
       title='Optional — point locations on synthetic raster')
ax.xaxis.set_major_locator(MaxNLocator(3))
ax.yaxis.set_major_locator(MaxNLocator(4))
ax.ticklabel_format(style='plain', useOffset=False)
# The colour bar explains the raster values, not the point labels.
fig.colorbar(image, ax=ax, label='Synthetic raster values', shrink=0.8)
plt.tight_layout()
plt.show()

**Self-check:** P2 belongs to row 2 / column 2 and samples 31. A more precise point coordinate does not create a finer-resolution raster value.

### A6. Zonal statistics — one summary per polygon

Z1 selects the upper-left 2 × 2 cells; Z2 selects the lower-right 2 × 2 cells. Predict Z1's valid-cell count and mean before running the code.

**Step 1 — Define two polygons.** Inspect their bounds to see which part of the grid each one covers.

In [ ]:
# Each rectangular Polygon will collect the raster cells inside its boundary.
zones = gpd.GeoDataFrame(
    {'zone_id': ['Z1', 'Z2']},
    geometry=[box(500000, 5339900, 500100, 5340000),
              box(500100, 5339800, 500200, 5339900)],
    crs='EPSG:32633'
)
print('Zone IDs:', zones['zone_id'].tolist())
print('Zone bounds (min x, min y, max x, max y):')
display(zones[['zone_id']].join(zones.geometry.bounds))

### A6 quick look — Optional: check the Polygon locations before masking

Plot the zone boundaries **before calculating their statistics**. The two polygons should cover different areas of the raster. This is a visual check of geometry and alignment, not a calculation of zonal means. **Optional: run or skip.**

In [ ]:
# OPTIONAL QUICK LOOK — overlay input Polygon boundaries on the teaching raster.
with rasterio.open(toy_raster_path) as src:
    raster_image = src.read(1, masked=True)
    raster_bounds = src.bounds

fig, ax = plt.subplots(figsize=(6, 5))
quick_cmap = plt.get_cmap('viridis').copy()
quick_cmap.set_bad('#dedede')
ax.imshow(raster_image,
          extent=(raster_bounds.left, raster_bounds.right,
                  raster_bounds.bottom, raster_bounds.top),
          origin='upper', cmap=quick_cmap, interpolation='nearest')

# GeoPandas draws the vector boundaries using the same map coordinates.
# This works because the raster and zones both use EPSG:32633.
zones.boundary.plot(ax=ax, color='#e34234', linewidth=2)
for _, zone in zones.iterrows():
    centre = zone.geometry.centroid
    ax.text(centre.x, centre.y, zone['zone_id'], ha='center', va='center',
            fontsize=12, weight='bold', color='#ffffff',
            bbox={'facecolor': '#ad392d', 'edgecolor': 'none', 'pad': 2})

ax.set(title='Optional — polygons before zonal statistics',
       xlabel='Easting (m)', ylabel='Northing (m)')
ax.xaxis.set_major_locator(MaxNLocator(3))
ax.yaxis.set_major_locator(MaxNLocator(4))
ax.ticklabel_format(style='plain', useOffset=False)
plt.tight_layout()
plt.show()


**Step 2 — Examine just Z1.** This makes it easier to understand what `mask()` selects before applying it to both zones.

In [ ]:
# First examine ONE polygon before running the loop for both zones.
z1 = zones.iloc[0]
with rasterio.open(toy_raster_path) as src:
    z1_clip, _ = mask(src, [mapping(z1.geometry)], crop=True, filled=False)

print('Z1 raster cells (masked means excluded / NoData):\n', z1_clip[0])
z1_values = z1_clip[0].compressed()
print('Valid Z1 values:', z1_values)
print('Z1 valid-cell count:', z1_values.size)

### A6 visual check — Optional: what did `mask()` keep for Z1?

Compare the full **4 × 4 raster** to the **2 × 2 crop for Z1**. The grey cell is NoData and **does not count toward the mean**. This plot visualizes the `z1_clip` already created above; it does not run the polygon analysis again. **Run or skip; no plotting code to write.**

In [ ]:
# OPTIONAL PLOT — inspect the actual masked raster created by rasterio.mask().
# The full surface has 16 cells; the Z1 clip has only the zone's 2 × 2 footprint.
full_grid = np.ma.masked_equal(surface_1, nodata)
zone_grid = z1_clip[0]  # This is already a masked array from the previous step.

# Missing cells are coloured grey, not treated as numeric -9999 values.
zone_colors = plt.get_cmap('viridis').copy()
zone_colors.set_bad('#d0d0d0')
fig, axes = plt.subplots(1, 2, figsize=(8.6, 4.3), constrained_layout=True)
for ax, (label, grid) in zip(
    axes, [('Full raster: 4 × 4', full_grid), ('Z1 crop: 2 × 2', zone_grid)]
):
    ax.imshow(grid, cmap=zone_colors, vmin=11, vmax=41,
              origin='upper', interpolation='nearest')
    # Write the values in each cell so the mean can be checked by hand.
    for row in range(grid.shape[0]):
        for col in range(grid.shape[1]):
            is_missing = bool(np.ma.getmaskarray(grid)[row, col])
            label_text = 'NoData' if is_missing else f'{grid[row, col]:g}'
            ax.text(col, row, label_text, ha='center', va='center',
                    color='black' if is_missing or grid[row, col] >= 30 else 'white',
                    fontsize=9)
    ax.set(title=label, xlabel='Column in this view', ylabel='Row in this view')
    ax.set_xticks(range(grid.shape[1]))
    ax.set_yticks(range(grid.shape[0]))
    ax.set_xticks(np.arange(-0.5, grid.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, grid.shape[0], 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=1)
    ax.tick_params(which='minor', length=0)
plt.show()
print('Z1 valid values used in the mean:', z1_values)

**Step 3 — Apply the same operation to Z1 and Z2.** The loop repeats the selection and summary once per polygon.

In [ ]:
# Each pass through the loop will add one summary dictionary to this list.
rows = []
with rasterio.open(toy_raster_path) as src:
    for _, zone in zones.iterrows():
        # mask() selects the cells inside the current polygon.
        clipped, _ = mask(src, [mapping(zone.geometry)],
                          crop=True, filled=False)
        values = clipped[0].compressed()  # Leave out masked cells.
        rows.append({
            'zone_id': zone['zone_id'],
            'count': int(values.size),
            'mean': float(values.mean()) if values.size else np.nan,
            'min': float(values.min()) if values.size else np.nan,
            'max': float(values.max()) if values.size else np.nan
        })
print('Number of zone summaries collected:', len(rows))

**Step 4 — Turn the summaries into a table and inspect them.** Compare both means with your calculations by hand.

In [ ]:
# A pandas DataFrame turns the collected dictionaries into a results table.
zonal_results = pd.DataFrame(rows)
display(zonal_results)
print('Z1 valid cells: 11, 13, 19 → mean:', (11 + 13 + 19) / 3)
print('Z2 valid cells: 31, 33, 39, 41 → mean:', (31 + 33 + 39 + 41) / 4)

### A6 visual check — Optional: which cells are inside each zone?

The coloured outlines are the Z1 and Z2 polygons. Compare their footprints to the grid and find Z1’s grey NoData cell, which does **not** contribute to its mean. **Run or skip; no plotting code to write.**

In [ ]:
# OPTIONAL PLOT — the zone statistics above are already complete.
# Both the raster and polygon outlines use the same coordinate system.
with rasterio.open(toy_raster_path) as src:
    bounds = src.bounds
    toy_pixels = src.read(1, masked=True)

# Give missing pixels a grey colour so they are not confused with measurements.
colors = plt.get_cmap('viridis').copy()
colors.set_bad('#d0d0d0')
fig, ax = plt.subplots(figsize=(6.8, 5.4))
# Put raster pixels in geographic position using their boundaries.
ax.imshow(toy_pixels, extent=(bounds.left, bounds.right, bounds.bottom, bounds.top),
          origin='upper', cmap=colors, interpolation='nearest')

# Draw each Shapely Polygon boundary as a line on top of the raster.
for (_, zone), outline_color in zip(zones.iterrows(), ['#d0442c', '#1b65b5']):
    x, y = zone.geometry.exterior.xy  # Boundary coordinates for drawing.
    ax.plot(x, y, color=outline_color, linewidth=3, label=zone['zone_id'])
    # Look up the mean already computed; plotting does NOT recalculate it.
    mean = zonal_results.set_index('zone_id').loc[zone['zone_id'], 'mean']
    # Place the zone ID and mean in the middle of each rectangle.
    ax.text(zone.geometry.centroid.x, zone.geometry.centroid.y,
            f"{zone['zone_id']}\nmean={mean:.2f}", ha='center', va='center',
            fontsize=9, color=outline_color, weight='bold',
            bbox={'facecolor':'white', 'alpha':0.8, 'edgecolor':'none'})

ax.set(xlabel='Easting (m, toy CRS)', ylabel='Northing (m, toy CRS)',
       title='Optional — zone boundaries over synthetic raster')
ax.xaxis.set_major_locator(MaxNLocator(3))
ax.yaxis.set_major_locator(MaxNLocator(4))
ax.ticklabel_format(style='plain', useOffset=False)
ax.legend(loc='upper right')  # Match outline colours to zone IDs.
plt.tight_layout()
plt.show()

**Self-check:** Z1 includes `[11, 13, 19, NoData]`, so its valid-cell mean is `43/3 ≈ 14.33`. Z2 includes `[31, 33, 39, 41]`, so its mean is `36`. This is a **continuous** synthetic raster; we will now switch to a **categorical** real raster.

### A7. What if a polygon covers only part of a cell?

Here we reuse the lecture's two-cell example: values **10** and **90**, with 100 m cells. The polygon covers the first cell and extends 10 m into the second. Predict which values the default rule will include, then compare it with `all_touched=True`.

In [ ]:
boundary_values = np.array([[10, 90]], dtype='float32')
boundary_transform = from_origin(600000, 5340000, 100, 100)
boundary_path = workdir / 'W05_Guided_Boundary.tif'
boundary_zone = box(600000, 5339900, 600110, 5340000)

with rasterio.open(boundary_path, 'w', driver='GTiff', width=2, height=1,
                   count=1, dtype='float32', crs='EPSG:32633',
                   transform=boundary_transform) as dst:
    dst.write(boundary_values, 1)


In [ ]:
with rasterio.open(boundary_path) as src:
    centre_clip, _ = mask(src, [mapping(boundary_zone)], crop=True,
                          filled=False, all_touched=False)
    touched_clip, _ = mask(src, [mapping(boundary_zone)], crop=True,
                           filled=False, all_touched=True)

centre_values = centre_clip[0].compressed()
touched_values = touched_clip[0].compressed()
print('Default rule values:', centre_values, 'mean:', centre_values.mean())
print('All-touched values:', touched_values, 'mean:', touched_values.mean())


**Explain the change.** The second cell's centre is outside the polygon, so the default rule keeps only 10. `all_touched=True` includes the second cell as well, giving a mean of 50. Each selected cell contributes a whole value: neither calculation weights it by the fraction of its area inside the polygon. In the Vienna examples below, we keep the default `all_touched=False` rule.

## Guided B — Real Vienna land cover

**Question:** What share of pixels in **one Vienna district** is classified as **Built-up** in ESA WorldCover 2021?

We reuse the same `mask()` and `.compressed()` steps with real categorical data. Land-cover IDs are categories, so we count the pixels for a class rather than taking the mean of its code. **Optional** maps are provided; no plotting code is required.

### B1. Load and inspect the two real datasets

**Step 1 — Check that the complete real-data files are present.** These two files should be right beside the Notebook. If they are missing, ask the instructor; do not substitute the older Week 3 or Week 4 clips.

In [ ]:
# Both full-city input files are stored beside this Notebook.
raster_path = data_dir / 'W05_Vienna_WorldCover2021_10m.tif'
district_path = data_dir / 'W05_Vienna_Districts_23.geojson'

print('Raster file found:', raster_path.exists())
print('District file found:', district_path.exists())
if not raster_path.exists() or not district_path.exists():
    raise FileNotFoundError('Week 5 full-city files are missing. Put both data files beside W05_Raster_VectorRaster_Integration_Lab.ipynb.')

**Step 2 — Inspect the raster metadata.** WorldCover stores land-cover class IDs, not elevation.

In [ ]:
# Read metadata before analysis: CRS, cell size and NoData tell us how to use the raster.
with rasterio.open(raster_path) as src:
    raster_crs = src.crs
    print('Raster CRS:', src.crs)
    print('Raster resolution:', src.res)
    print('Number of bands:', src.count)
    print('Raster NoData:', src.nodata)
    print('Raster bounds:', src.bounds)
    # These metadata values help us check that the correct data was supplied.
    if src.count != 1 or src.nodata != 0:
        raise ValueError('Unexpected raster format. Please check the Week 5 data files.')
    if src.crs is None or src.crs.to_epsg() != 31256 or not np.allclose(src.res, (10, 10), rtol=0, atol=1e-6):
        raise ValueError('This is not the prepared 10 m EPSG:31256 Week 5 raster.')

### B1 quick look — Optional: inspect the raster's geographic coverage

We have checked **CRS, resolution, bounds and NoData**. Now plot a quick preview to see **where valid WorldCover pixels exist**. Dark cells have a land-cover category; light cells are outside the valid data area. This is a *coverage check*, not yet a map of Tree Cover or Built-up. **Optional: plotting code is complete.**

In [ ]:
# OPTIONAL QUICK LOOK — draw a fast overview immediately after loading the raster.
from rasterio.enums import Resampling
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

with rasterio.open(raster_path) as src:
    # Read a smaller display array to keep plotting responsive.
    # Nearest neighbour preserves categorical values; calculations use the full raster.
    preview_factor = min(1.0, 650 / max(src.width, src.height))
    preview_size = (max(1, int(src.height * preview_factor)),
                    max(1, int(src.width * preview_factor)))
    preview = src.read(1, out_shape=preview_size, resampling=Resampling.nearest)
    raster_bounds = src.bounds
    nodata_value = src.nodata

# For this first visual check, show valid data (1) versus NoData (0) only.
coverage = (preview != nodata_value).astype('uint8')
fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.imshow(coverage,
          extent=(raster_bounds.left, raster_bounds.right,
                  raster_bounds.bottom, raster_bounds.top),
          origin='upper', cmap=ListedColormap(['#f2f3f5', '#688d91']),
          vmin=0, vmax=1, interpolation='nearest')
ax.legend(handles=[Patch(facecolor='#688d91', label='Valid land-cover pixels'),
                   Patch(facecolor='#f2f3f5', label='NoData / outside Vienna')],
          loc='lower left', framealpha=0.9)
ax.set(title='Optional — WorldCover data coverage (not class values)',
       xlabel='Easting (m, EPSG:31256)', ylabel='Northing (m, EPSG:31256)')
ax.ticklabel_format(style='plain', useOffset=False)
plt.tight_layout()
plt.show()


**Step 3 — Read the administrative districts.** Compare their CRS with the raster before combining the two layers.

In [ ]:
# GeoPandas reads the polygons and their attribute columns from GeoJSON.
districts = gpd.read_file(district_path)
print('Districts loaded:', len(districts))
print('District CRS before conversion:', districts.crs)
display(districts[['district_id', 'district_name']].head())

### B1 quick look — Optional: plot the districts before analysis

After `gpd.read_file()`, inspect the attribute table **and draw the polygons once**. Check that they form a complete city outline and are not empty or scattered. This single-layer plot does not require the raster CRS yet; matching CRS matters when we **overlay** the two layers next. **Optional: run or skip.**

In [ ]:
# OPTIONAL QUICK LOOK — view the district geometries as soon as they are loaded.
fig, ax = plt.subplots(figsize=(7, 6))
# plot() uses the GeoDataFrame's own geometry and its current coordinates.
districts.plot(ax=ax, facecolor='#dce9db', edgecolor='#4c6751',
               linewidth=0.65)
ax.set(title='Optional — Vienna district polygons (before CRS matching)',
       xlabel='X coordinate (district CRS)',
       ylabel='Y coordinate (district CRS)')
plt.tight_layout()
plt.show()


**Step 4 — Match the CRS and check the coverage list.** The full-city data should contain the 23 unique district IDs.

In [ ]:
# Convert vector coordinates to the raster's coordinate system.
# Matching the CRS is essential before clipping raster cells with polygons.
districts = districts.to_crs(raster_crs)
print('District CRS after conversion:', districts.crs)
print('Number of districts:', len(districts))
print('Unique district IDs:', districts['district_id'].nunique())
print('District IDs:', sorted(districts['district_id'].astype(int).tolist()))
if len(districts) != 23 or set(districts['district_id']) != set(range(1, 24)):
    raise ValueError('Expected 23 unique districts. Please check the full Vienna district file.')

**What does a pixel value mean?** These are WorldCover **category IDs**, not heights or NDVI. The data year is 2021. The teacher has prepared a projected 10 m raster using **nearest-neighbour** resampling to keep categorical codes unchanged.

In [ ]:
# ESA WorldCover stores category IDs; the integers are labels, not quantities.
CLASS_LABELS = {
    10: 'Tree Cover', 20: 'Shrubland', 30: 'Grassland',
    40: 'Cropland', 50: 'Built-up', 60: 'Bare / sparse vegetation',
    70: 'Snow and Ice', 80: 'Permanent water bodies',
    90: 'Herbaceous wetland', 95: 'Mangroves',
    100: 'Moss and lichen'
}
NODATA_CODE = 0
BUILTUP_CODE = 50
print('Built-up class code:', BUILTUP_CODE)
print('Tree Cover code in the legend:', [code for code, name in CLASS_LABELS.items() if name == 'Tree Cover'])

### B1 visual check — Optional: see real Vienna land cover and district boundaries

This overview displays a **downsampled view for speed only**. The categories and district boundaries remain in their map coordinates. **All statistics later use the full-resolution original**, not the overview. **Run or skip; no plotting code to write.**

In [ ]:
# OPTIONAL MAP — pre-written for exploration; skip it if you prefer.
from rasterio.enums import Resampling
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

# The full raster is large, so read a smaller preview for quicker drawing.
# Nearest-neighbour keeps the original class IDs instead of mixing categories.
# IMPORTANT: later percentage calculations still use the full-resolution raster.
with rasterio.open(raster_path) as src:
    factor = min(1.0, 750 / max(src.height, src.width))
    preview = src.read(1, out_shape=(max(1, int(src.height * factor)),
                                     max(1, int(src.width * factor))),
                       resampling=Resampling.nearest)
    bounds = src.bounds

# Group original codes into five DISPLAY categories; this does not edit the data.
# 0=NoData, 1=Tree, 2=Built-up, 3=Water, 4=Other land-cover classes.
category_view = np.full(preview.shape, 4, dtype='uint8')
for code, display_code in [(0, 0), (10, 1), (50, 2), (80, 3)]:
    category_view[preview == code] = display_code

# A ListedColormap assigns one fixed colour to each discrete category.
colours = ['#ffffff', '#348653', '#c96d68', '#4a9bd1', '#d2d0c9']
cmap = ListedColormap(colours)
norm = BoundaryNorm(np.arange(-0.5, 5.5, 1), cmap.N)

fig, ax = plt.subplots(figsize=(9, 7))
# The raster and district outlines use the same projected map coordinates.
ax.imshow(category_view,
          extent=(bounds.left, bounds.right, bounds.bottom, bounds.top),
          origin='upper', cmap=cmap, norm=norm, interpolation='nearest')
districts.boundary.plot(ax=ax, color='#313a43', linewidth=0.6)

# Explain the class colours with legend patches rather than a numeric colour bar.
ax.legend(handles=[Patch(facecolor=colours[i], label=label)
                   for i, label in [(1, 'Tree Cover'), (2, 'Built-up'),
                                    (3, 'Water'), (4, 'Other classes')]],
          loc='lower left', framealpha=0.92)
ax.set(title='Optional — Vienna WorldCover 2021 and 23 district boundaries',
       xlabel='Easting (m, EPSG:31256)', ylabel='Northing (m, EPSG:31256)')
ax.ticklabel_format(style='plain', useOffset=False)
plt.tight_layout()
plt.show()

### B2. Guided example — Built-up share in one district

Use district **2 (Leopoldstadt)** as the worked example. We do not assume its actual percentage in advance; we compute it from the supplied 2021 raster. Compare the district geometry with the raster CRS before extracting cells.

**Step 1 — Select Leopoldstadt (district 2).** Look at the selected feature before using its geometry.

In [ ]:
# Select the one polygon whose Built-up share we will calculate first.
guided_district = districts.loc[districts['district_id'] == 2].iloc[0]
print('Selected district:', guided_district['district_name'])
print('Geometry type:', guided_district.geometry.geom_type)

**Step 2 — Clip WorldCover to that district.** The masked array keeps the distinction between selected cells, excluded cells, and NoData.

In [ ]:
# mask() returns both cropped cells and the transform for that cropped area.
with rasterio.open(raster_path) as src:
    guided_clip, guided_transform = mask(
        src, [mapping(guided_district.geometry)], crop=True, filled=False
    )
print('Clipped array shape (bands, rows, columns):', guided_clip.shape)

**Step 3 — Look at the valid category values.** Notice that `compressed()` removes masked cells.

In [ ]:
# compressed() drops masked cells so we only count valid class labels.
guided_values = guided_clip[0].compressed()
valid_count = int(guided_values.size)
print('Number of valid pixels in this district:', valid_count)
print('A few class codes:', guided_values[:20])

**Step 4 — Count Built-up and calculate the percentage.** The denominator is the number of valid pixels, not the entire rectangle around the polygon.

In [ ]:
# WorldCover class 50 represents Built-up.
# Count class-50 cells; do not take the mean of categorical codes.
builtup_count = int(np.count_nonzero(guided_values == BUILTUP_CODE))
print('Built-up pixels:', builtup_count)
print('Valid pixels:', valid_count)

# Divide by ALL valid cells in the district, not by its rectangular bounding box.
if valid_count > 0:
    builtup_pct = 100 * builtup_count / valid_count
    print('Built-up share (%):', round(builtup_pct, 2))
else:
    builtup_pct = None
    print('No valid pixels were found. Check the district and raster coverage.')

**Guided verification:** The denominator is **all valid land-cover pixels inside this district**, not all cells in the rectangular cropped array. Why would dividing by the total raster dimensions be wrong? What would a mean of category codes mean?

### B3. Optional — see Built-up pixels in the Guided district

The shaded cells below are those classified as **Built-up (50)**, displayed inside the selected district outline. All other **valid** cells appear in light grey; NoData and cells outside the district are blank. This is **only a visualisation** — it is not the percentage calculation. **Run or skip; no plotting code to write.**

In [ ]:
# OPTIONAL MAP — highlights class 50 in just the selected district.
from matplotlib.colors import ListedColormap
from rasterio.transform import array_bounds

# The earlier mask() call returned the district's cropped array and its transform.
selected_pixels = guided_clip[0]
height, width = selected_pixels.shape
map_west, map_south, map_east, map_north = array_bounds(height, width, guided_transform)
show_extent = (map_west, map_east, map_south, map_north)

# First layer: all valid land-cover cells (light grey).
# Masked/NoData cells stay blank, outside the polygon is also blank.
valid_footprint = np.ma.masked_array(
    np.ones((height, width), dtype='uint8'),
    mask=np.ma.getmaskarray(selected_pixels)
)
# Second layer: display ONLY Built-up cells (red). Other classes are hidden.
builtup_footprint = np.ma.masked_where(
    selected_pixels != BUILTUP_CODE, np.ones((height, width), dtype='uint8')
)

fig, ax = plt.subplots(figsize=(7.5, 6))
# Draw the light-grey background first; the red Built-up cells go on top.
ax.imshow(valid_footprint, extent=show_extent, origin='upper',
          cmap=ListedColormap(['#e2e6e8']), interpolation='nearest')
ax.imshow(builtup_footprint, extent=show_extent, origin='upper',
          cmap=ListedColormap(['#bb655c']), interpolation='nearest')

# Add the district outline as a vector layer to show the analysis boundary.
gpd.GeoSeries([guided_district.geometry], crs=districts.crs).boundary.plot(
    ax=ax, edgecolor='#27343f', linewidth=1.2)
ax.set(title=f"Optional — {guided_district['district_name']}: Built-up pixels (2021)",
       xlabel='Easting (m, EPSG:31256)', ylabel='Northing (m, EPSG:31256)')
ax.ticklabel_format(style='plain', useOffset=False)
plt.tight_layout()
plt.show()

## Independent Checkpoint — Tree Cover in all 23 districts

**Research question:** *How does the proportion of WorldCover 2021 pixels classified as Tree Cover vary across Vienna's 23 districts?*

Use the same files and the `mask()` and `.compressed()` concepts from Guided B, but **transfer** the workflow from Built-up to Tree Cover and apply it to all 23 districts.

- **C1–C2:** Complete the code and create one result per district.
- **C3:** Inspect the table, check the values, and verify one percentage.
- **C4 (Optional):** Check your table, then explore your results with the ready-made map and charts. All plotting code is provided.
- **C5:** Use *your own results* to describe geographic differences, compare districts, and suggest a possible explanation.

The map and charts are **Optional**. You do not need to write or run plotting code to answer C5; the district results table contains the required numbers.


### Task C1 — Select the Tree Cover class

In [ ]:
# TODO C1: Look up the WorldCover code for Tree Cover in CLASS_LABELS.
TREE_CODE = ...
print('Selected Tree Cover class code:', TREE_CODE)
print('Selected class name:', CLASS_LABELS.get(TREE_CODE, 'Choose a class code'))
print('Is this Tree Cover?', CLASS_LABELS.get(TREE_CODE) == 'Tree Cover')


### Task C2 — One result per district

Complete the TODOs using the structure from B2. The loop is provided because loops were introduced earlier; the **spatial extraction, class counting, percentage and result-record construction** are yours to adapt. Use `src` as a Rasterio dataset. `values.size` must count only valid, unmasked pixels. Keep the same default pixel-selection rule (`all_touched=False`) as Guided B.

In [ ]:
# Start an empty list: each district should contribute one result row.
records = []
with rasterio.open(raster_path) as src:
    for _, district in districts.iterrows():
        # TODO C2a: clip this district using mask(..., crop=True, filled=False).
        clipped, _ = ...

        # TODO C2b: keep only unmasked pixel values from band 1.
        values = ...

        # TODO C2c: count valid cells and those classified as Tree Cover.
        valid_pixels = ...
        tree_pixels = ...

        # TODO C2d: calculate the tree share in percent (guard against zero valid cells).
        tree_pct = ...

        # TODO C2e: append a dictionary with the five fields below.
        # district_id, district_name, valid_pixels, tree_pixels, tree_cover_pct
        records.append(...)

print('Result records collected:', len(records))

**Inspect your records.** If the count is not 23, return to Task C2 and check that one row is added for each district.

In [ ]:
# Inspect the table shape before trusting any district ranking.
checkpoint_results = pd.DataFrame(records)
print('Table shape (rows, columns):', checkpoint_results.shape)
display(checkpoint_results.head())

### Task C3 — Inspect the completed table

**Read the checks one at a time.** A `True` result is encouraging; `False` means you should revisit your calculation or the corresponding district. These checks do not calculate your answer for you.

In [ ]:
# Checking the result table is part of the geographic analysis.
required_columns = ['district_id', 'district_name', 'valid_pixels',
                    'tree_pixels', 'tree_cover_pct']
missing_columns = [name for name in required_columns
                   if name not in checkpoint_results.columns]
print('Missing columns:', missing_columns)
print('Result rows:', len(checkpoint_results))
print('Expected rows:', 23)
print('Correct number of rows?', len(checkpoint_results) == 23)

**Check the values only after the column names are correct.** Look for duplicate districts, missing values, counts and percentage ranges.

In [ ]:
if missing_columns:
    print('Complete the missing columns in Task C2 before checking the values.')
else:
    ids = pd.to_numeric(checkpoint_results['district_id'], errors='coerce')
    counts = checkpoint_results[['valid_pixels', 'tree_pixels']].apply(
        pd.to_numeric, errors='coerce')
    percentages = pd.to_numeric(checkpoint_results['tree_cover_pct'], errors='coerce')
    print('District IDs present and numeric?', ids.notna().all())
    print('District IDs unique?', ids.is_unique)
    print('All 23 exact district IDs present?', set(ids) == set(range(1, 24)))
    print('Pixel counts present and numeric?', np.isfinite(counts).all().all())
    print('Pixel counts are whole numbers?', (counts % 1 == 0).all().all())
    print('All districts have valid pixels?', (counts['valid_pixels'] > 0).all())
    print('Tree counts are not negative?', (counts['tree_pixels'] >= 0).all())
    print('Tree count does not exceed valid count?',
          (counts['tree_pixels'] <= counts['valid_pixels']).all())
    print('All percentages between 0 and 100?', percentages.between(0, 100).all())
    print('Correct Tree Cover class selected?', CLASS_LABELS.get(TREE_CODE) == 'Tree Cover')


**Inspect the rankings.** The following cell displays the highest and lowest percentages once the table contains the expected columns.

In [ ]:
if missing_columns:
    print('Cannot rank districts yet; complete Task C2 first.')
else:
    print('Highest Tree Cover shares:')
    display(checkpoint_results.sort_values('tree_cover_pct', ascending=False).head())
    print('Lowest Tree Cover shares:')
    display(checkpoint_results.sort_values('tree_cover_pct').head())

**Verify one district against the original raster.** The next cell selects a district other than the guided example, extracts its pixels again, and compares the new counts and percentage with your table. This can reveal an incorrect selection or count even when a percentage calculated from your stored counts looks plausible.

In [ ]:
if missing_columns or checkpoint_results.empty:
    print('Finish Task C2 and fix the result columns before this check.')
elif CLASS_LABELS.get(TREE_CODE) != 'Tree Cover':
    print('Check the selected class in Task C1 before verifying Tree Cover.')
else:
    ids = pd.to_numeric(checkpoint_results['district_id'], errors='coerce')
    check_rows = checkpoint_results.loc[ids.isin(districts['district_id']) & (ids != 2)]
    if check_rows.empty:
        print('No valid district other than district 2 is available to check.')
    else:
        row = check_rows.iloc[0]
        district_id = float(row['district_id'])
        zone = districts.loc[districts['district_id'] == district_id].iloc[0]
        with rasterio.open(raster_path) as src:
            checked_clip, _ = mask(src, [mapping(zone.geometry)], crop=True, filled=False)
        checked_values = checked_clip[0].compressed()
        checked_valid = int(checked_values.size)
        checked_tree = int(np.count_nonzero(checked_values == TREE_CODE))
        if checked_valid == 0:
            print('No valid pixels in the selected district. Check the input data.')
        else:
            checked_pct = 100 * checked_tree / checked_valid
            stored = pd.to_numeric(row[['valid_pixels', 'tree_pixels', 'tree_cover_pct']],
                                   errors='coerce')
            print('District checked:', zone['district_name'])
            print('Valid-pixel count matches?', stored['valid_pixels'] == checked_valid)
            print('Tree-pixel count matches?', stored['tree_pixels'] == checked_tree)
            print('Percentage from the raster:', round(checked_pct, 2))
            print('Percentage in your table:', stored['tree_cover_pct'])
            print('Percentages agree within 0.01 percentage points?',
                  np.isclose(stored['tree_cover_pct'], checked_pct, rtol=0, atol=0.01))


### Task C4 — Optional: explore your results with a map and charts

The map and charts below are **Optional**. All plotting code is already written for you. You can use the C3 results table instead if you prefer not to run the plots.

**First, check the input variables.** Run the next cell before any map or chart. It prints clear messages when a result table is missing or incomplete. Only move on to the figures when it says **Ready to plot**.

In [ ]:
# OPTIONAL — CHECK INPUTS BEFORE ANY MAP OR CHART.
# The plots use the district statistics you created in Task C2.
chart_ready = False
map_ready = False

# This first check prevents a NameError if Task C2 has not been run.
if 'pd' not in globals() or 'gpd' not in globals() or 'plt' not in globals():
    print('Run the Setup import cells before using the optional plots.')
elif 'checkpoint_results' not in globals():
    print('No checkpoint_results found. Run Tasks C1–C3 first.')
elif not isinstance(checkpoint_results, pd.DataFrame) or checkpoint_results.empty:
    print('Your result table is empty. Complete Task C2 first.')
else:
    needed = ['district_id', 'district_name', 'valid_pixels',
              'tree_pixels', 'tree_cover_pct']
    missing = [col for col in needed if col not in checkpoint_results.columns]
    print('Rows in result table:', len(checkpoint_results))
    print('Missing columns:', missing)

    if missing:
        print('Fix the missing columns in Task C2 before plotting.')
    else:
        # Make a separate copy for charts; do not change the student's table.
        chart_df = checkpoint_results[needed].copy()
        for col in ['district_id', 'valid_pixels', 'tree_pixels', 'tree_cover_pct']:
            chart_df[col] = pd.to_numeric(chart_df[col], errors='coerce')

        # Check the main things needed for sensible geographic charts.
        problems = []
        if len(chart_df) != 23:
            problems.append('Expected 23 district rows.')
        if chart_df['district_id'].isna().any() or chart_df['district_id'].duplicated().any():
            problems.append('District IDs must be present and unique.')
        elif set(chart_df['district_id']) != set(range(1, 24)):
            problems.append('District IDs should cover all districts from 1 to 23.')
        if chart_df['district_name'].isna().any():
            problems.append('Some district names are missing.')
        if chart_df[['valid_pixels', 'tree_pixels', 'tree_cover_pct']].isna().any().any():
            problems.append('Some pixel counts or percentages are missing or not numeric.')
        elif ((chart_df['valid_pixels'] <= 0).any()
              or (chart_df['tree_pixels'] < 0).any()
              or (chart_df['tree_pixels'] > chart_df['valid_pixels']).any()):
            problems.append('Some pixel counts are not valid.')
        elif not chart_df['tree_cover_pct'].between(0, 100).all():
            problems.append('Tree Cover percentages must be between 0 and 100.')
        else:
            # Confirm the plotted percentage matches the pixel counts.
            recalculated = 100 * chart_df['tree_pixels'] / chart_df['valid_pixels']
            if ((chart_df['tree_cover_pct'] - recalculated).abs() > 0.15).any():
                problems.append('Some percentages do not match the pixel counts.')

        if problems:
            print('Not ready to plot. Please check:')
            for problem in problems:
                print('-', problem)
        else:
            chart_df = chart_df.sort_values('tree_cover_pct').reset_index(drop=True)
            chart_df['district_id'] = chart_df['district_id'].astype(int)
            chart_ready = True
            print('Ready to plot: the 23 district percentages look valid.')
            display(chart_df.head())

# A map also needs the district polygons, not just the results table.
if chart_ready:
    if 'districts' not in globals() or not isinstance(districts, gpd.GeoDataFrame):
        print('Map not ready: load the district GeoDataFrame in Guided B first.')
    elif 'district_id' not in districts.columns or len(districts) != 23:
        print('Map not ready: check the 23 district polygons and their IDs.')
    else:
        district_ids = pd.to_numeric(districts['district_id'], errors='coerce')
        if (district_ids.isna().any() or district_ids.duplicated().any()
                or set(district_ids) != set(range(1, 24))):
            print('Map not ready: district IDs must be unique numbers from 1 to 23.')
        elif districts.geometry.isna().any() or districts.geometry.is_empty.any():
            print('Map not ready: some district polygons are missing or empty.')
        else:
            map_ready = True
            print('Map input ready: district polygons are available.')


**Optional map — Where are the high and low Tree Cover districts?**

A choropleth uses the district boundaries to show where the Tree Cover percentages are higher or lower. Read the map alongside the table: a large polygon is not automatically a district with a larger **percentage**.

In [ ]:
# OPTIONAL MAP — run the input check above first.
if not map_ready:
    print('Map skipped. Fix the printed input checks above, then run this cell again.')
else:
    # The merge joins your percentage table to the correct district geometry.
    map_districts = districts.copy()
    map_districts['district_id'] = pd.to_numeric(
        map_districts['district_id'], errors='coerce'
    )
    checkpoint_map = map_districts.merge(
        chart_df[['district_id', 'tree_cover_pct']],
        on='district_id', how='left', validate='one_to_one'
    )
    print('Districts without a matched percentage:',
          checkpoint_map['tree_cover_pct'].isna().sum())

    # One colour per district; legend describes the percentage scale.
    fig, ax = plt.subplots(figsize=(9, 7))
    checkpoint_map.plot(
        column='tree_cover_pct', cmap='YlGn', vmin=0, vmax=100,
        edgecolor='#424242', linewidth=0.55,
        legend=True,
        legend_kwds={'label': 'Pixels classified as Tree Cover (%)', 'shrink': 0.65},
        ax=ax
    )
    ax.set_title('Vienna districts — Tree Cover classified pixels, 2021')
    ax.set_axis_off()
    fig.tight_layout()
    plt.show()


**Optional chart 1 — How do all 23 districts rank?**

This sorted bar chart makes it easy to identify the highest and lowest percentages (C5a). It also helps you compare **how far apart** the percentages are (C5b).

In [ ]:
# OPTIONAL CHART 1 — all district percentages, sorted from low to high.
if not chart_ready:
    print('Chart skipped. Run the input check above and fix the result table first.')
else:
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.barh(chart_df['district_name'], chart_df['tree_cover_pct'])
    ax.set_xlabel('Pixels classified as Tree Cover (%)')
    ax.set_ylabel('District')
    ax.set_title('Tree Cover share — ranked Vienna districts')
    ax.grid(axis='x', alpha=0.25)
    fig.tight_layout()
    plt.show()


**Optional chart 2 — Are district percentages similar or different?**

A histogram shows the **distribution** across 23 districts. Each bar counts **districts**, not raster pixels. Use this chart to think about whether the values are clustered or spread out (C5b).

In [ ]:
# OPTIONAL CHART 2 — each histogram bar counts districts in a % range.
if not chart_ready:
    print('Chart skipped. Run the input check above and fix the result table first.')
else:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.hist(chart_df['tree_cover_pct'], bins=8, edgecolor='white')
    ax.set_xlabel('Pixels classified as Tree Cover (%)')
    ax.set_ylabel('Number of districts')
    ax.set_title('Distribution of Tree Cover shares across Vienna')
    ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    plt.show()


**Optional chart 3 — Which contrasts are most striking?**

This chart shows the five districts at each end of the ranking. It can help you choose a contrast to discuss in C5c. A chart can show **what** differs; the geographic explanation still needs your reasoning.

In [ ]:
# OPTIONAL CHART 3 — compare the lowest five and highest five districts.
if not chart_ready:
    print('Chart skipped. Run the input check above and fix the result table first.')
else:
    bottom_five = chart_df.head(5)
    top_five = chart_df.tail(5)
    comparison = pd.concat([bottom_five, top_five]).sort_values('tree_cover_pct')

    fig, ax = plt.subplots(figsize=(9, 5.8))
    ax.barh(comparison['district_name'], comparison['tree_cover_pct'])
    ax.set_xlabel('Pixels classified as Tree Cover (%)')
    ax.set_ylabel('District')
    ax.set_title('Five lowest and five highest Tree Cover shares')
    ax.grid(axis='x', alpha=0.25)
    fig.tight_layout()
    plt.show()


### Task C5 — Read, compare and explain your results (required)

Now use **your own 23-district results** to answer the following questions. The C4 map and charts can help you notice patterns, but they are **Optional**. You can also answer using the C3 results table. Give short answers supported by your numbers.

**C5a — What stands out?** Which district has the highest Tree Cover share and which has the lowest? Give their percentages and the difference between them in **percentage points**.

**Your answer:**


**C5b — How much do districts vary?** Looking beyond the highest and lowest values, do Tree Cover percentages seem quite similar or noticeably different across Vienna? Support your observation with the percentages of **two other districts**.

**Your answer:**


**C5c — What might explain a difference?** Choose two districts with clearly different Tree Cover percentages. Suggest **one possible geographic explanation**, such as differences in land use or urban form. What additional map or dataset could help you check your idea? Remember that your suggestion is a **hypothesis**, not proof.

**Your answer:**


## Submission check

Before submitting according to Moodle instructions, confirm that you have:

- Run all **Guided A** computations and retained their predicted/checked outputs: alignment, mosaic, difference, point sampling, and zonal statistics.
- Run **Guided B** using authentic 2021 land cover for one district and understood how Built-up % was computed.
- Completed all **Checkpoint C1–C3** TODOs for 23 district Tree Cover percentages.
- **Optional:** run any of the **pre-written Guided plots** to inspect raster grids, point locations, zone boundaries or the Vienna map; you can skip all plotting cells.
- **Optional:** first check the plot inputs, then use the **provided** Checkpoint map and/or charts in C4 to interpret your results. Plotting is not assessed, and no code editing is required.
- Re-extracted one district from the original raster and checked its counts and percentage in **C3** and answered the three **C5 interpretation questions** in your own words.

**Notebook outputs:** the six tiny `W05_Guided_*.tif` files are synthetic teaching outputs: three tile files, two surface files, and one boundary-example file. They can be deleted after the lab. The two supplied Vienna input files stay beside the notebook; no extra data folder or map export is required.